# Know your customers 🛍️

One of the most common uses of K-Means is to get to know your customers. A shopping mall wants to understand who its customers are, to adapt its marketing to each group. In this exercise, you build customer segments, choose their number, describe them and turn them into a recommendation.

## The dataset 🧾

`data/mall_customers.csv` is a small teaching dataset of 200 fictional customers. For each customer, it gives:

* `CustomerID`: an identifier;
* `Genre`: the gender (`Male` / `Female`);
* `Age`;
* `Annual Income (k$)`: the annual income, in thousands of dollars;
* `Spending Score (1-100)`: a score given by the mall from the customer's purchases (100 = spends a lot).

The libraries and the data are loaded for you.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

dataset = pd.read_csv("data/mall_customers.csv")
dataset.head()

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


## Exploration 🔍

1. How many customers are there? Are there missing values? Describe the numerical variables and count the customers of each gender.

In [ ]:
print("shape:", dataset.shape)
print("missing values:", dataset.isna().sum().sum())
print(dataset["Genre"].value_counts(), "\n")
dataset.describe().round(1)

shape: (200, 5)
missing values: 0
Genre
Female    112
Male       88
Name: count, dtype: int64 



,CustomerID,Age,Annual Income (k$),Spending Score (1-100)
count,200.0,200.0,200.0,200.0
mean,100.5,38.8,60.6,50.2
std,57.9,14.0,26.3,25.8
min,1.0,18.0,15.0,1.0
25%,50.8,28.8,41.5,34.8
50%,100.5,36.0,61.5,50.0
75%,150.2,49.0,78.0,73.0
max,200.0,70.0,137.0,99.0


**200 customers, no missing value. Ages range from 18 to 70, incomes from 15 to 137 k\$, spending scores from 1 to 99. There are 112 women and 88 men.**

2. Draw the customers in the plane (annual income, spending score). How many groups can you see by eye?

In [ ]:
fig = px.scatter(dataset, x="Annual Income (k$)", y="Spending Score (1-100)", title="The 200 customers")
fig.show()

**Five groups appear: a dense group in the middle (average income, average spending), and four groups in the corners: low income with low spending, low income with high spending, high income with low spending, high income with high spending.**

## Preparing the data ⚙️

We want segments of customers **as buyers**, so we cluster them on two variables: `Annual Income (k$)` and `Spending Score (1-100)`. We keep `Age` and `Genre` aside, to describe the segments afterwards.

3. Why should `CustomerID` never be used for clustering? Put the two variables in a DataFrame `X`, then standardise them into `X_scaled`. Why is it useful here, although both variables have a similar range?

💡 Hint: compare the standard deviations of the two variables. And think about what would happen if the income was given in dollars instead of thousands of dollars.

In [ ]:
X = dataset[["Annual Income (k$)", "Spending Score (1-100)"]]
print(X.std().round(1))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Annual Income (k$)        26.3
Spending Score (1-100)    25.8
dtype: float64


**`CustomerID` is only a row number: two customers with close identifiers are not similar in any way, so a distance on it means nothing. The two variables have similar standard deviations (26.3 and 25.8), so scaling changes little here. But K-Means depends on the units: in dollars instead of thousands of dollars, the income would be 1,000 times larger and would decide everything alone. Scaling makes the result independent of the units. It is a good habit.**

## How many segments? 🔢

4. Use the elbow method: fit K-Means for $K$ from 1 to 10 (with `n_init=10, random_state=0`), store the inertia of each fit and plot it. Which $K$ would you choose?

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_scaled).inertia_ for k in ks]
for k, inertia in zip(ks, inertias):
    print(f"K={k:>2}: inertia {inertia:6.1f}")
fig = px.line(x=list(ks), y=inertias, markers=True, title="Elbow method")
fig.update_layout(xaxis_title="number of clusters K", yaxis_title="inertia")
fig.show()

K= 1: inertia  400.0
K= 2: inertia  269.3
K= 3: inertia  157.7
K= 4: inertia  108.9
K= 5: inertia   65.6
K= 6: inertia   55.1
K= 7: inertia   44.9
K= 8: inertia   37.1
K= 9: inertia   32.3
K=10: inertia   29.2


**The inertia drops a lot until $K=5$ (65.6), then it decreases much more slowly (55.1 for $K=6$, 44.9 for $K=7$): the elbow is at $K=5$.**

5. Check with the silhouette score, for $K$ from 2 to 10. Does it confirm your choice?

💡 Hint: `silhouette_score(X_scaled, labels)` needs the cluster of each customer, given by `fit_predict`.

In [ ]:
ks = range(2, 11)
sil = [silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_scaled)) for k in ks]
for k, s in zip(ks, sil):
    print(f"K={k:>2}: silhouette score {s:.3f}")
fig = px.bar(x=list(ks), y=sil, title="Silhouette score for each K")
fig.update_layout(xaxis_title="number of clusters K", yaxis_title="silhouette score")
fig.show()

K= 2: silhouette score 0.286
K= 3: silhouette score 0.467
K= 4: silhouette score 0.494
K= 5: silhouette score 0.555
K= 6: silhouette score 0.538
K= 7: silhouette score 0.526
K= 8: silhouette score 0.457
K= 9: silhouette score 0.458
K=10: silhouette score 0.448


**Yes: the silhouette score is highest for $K=5$ (0.555). The two methods agree, and so does our eye (question 2). We take $K=5$.**

6. Fit K-Means with the $K$ you chose. Add the cluster of each customer to `dataset`, in a column `cluster`, and count the customers of each cluster.

In [ ]:
kmeans = KMeans(n_clusters=5, n_init=10, random_state=0)
dataset["cluster"] = kmeans.fit_predict(X_scaled)
dataset["cluster"].value_counts().sort_index()

cluster
0    81
1    39
2    35
3    23
4    22
Name: count, dtype: int64

## Interpreting the segments 🔎

7. Draw the customers in the plane (annual income, spending score), coloured by cluster, and add the centres of the clusters in the original units.

💡 Hint: `scaler.inverse_transform(kmeans.cluster_centers_)` brings the centres back to the original units. Convert the `cluster` column to text (`astype(str)`) to get one colour per cluster.

In [ ]:
centres = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=X.columns)
fig = px.scatter(dataset, x="Annual Income (k$)", y="Spending Score (1-100)", color=dataset["cluster"].astype(str),
                 category_orders={"color": [str(k) for k in range(5)]}, labels={"color": "cluster"},
                 title="The 5 customer segments")
fig.add_trace(go.Scatter(x=centres.iloc[:, 0], y=centres.iloc[:, 1], mode="markers", name="centres",
                         marker=dict(symbol="x", size=14, color="#111111")))
fig.show()

8. Build a profile table: for each cluster, the number of customers, the mean age, the mean income, the mean spending score and the share of women. Which variables were **not** used to build the clusters?

In [ ]:
profile = dataset.groupby("cluster").agg(
    customers=("CustomerID", "size"),
    mean_age=("Age", "mean"),
    mean_income=("Annual Income (k$)", "mean"),
    mean_spending=("Spending Score (1-100)", "mean"),
    share_of_women=("Genre", lambda g: (g == "Female").mean()),
).round(2)
profile

,customers,mean_age,mean_income,mean_spending,share_of_women
cluster,,,,,
0,81,42.72,55.30,49.52,0.59
1,39,32.69,86.54,82.13,0.54
2,35,41.11,88.20,17.11,0.46
3,23,45.22,26.30,20.91,0.61
4,22,25.27,25.73,79.36,0.59


**The age and the gender were not used by K-Means, yet they help to describe the segments. The share of women is close to the global share (56%) in every cluster, between 0.46 and 0.61: the gender does not really separate the segments. The age does: the two high-spending clusters are young (mean ages 25.3 and 32.7), while the high-income, low-spending cluster is older (41.1).**

9. Draw the customers in the plane (age, spending score), coloured by cluster. What do you notice?

In [ ]:
fig = px.scatter(dataset, x="Age", y="Spending Score (1-100)", color=dataset["cluster"].astype(str),
                 category_orders={"color": [str(k) for k in range(5)]}, labels={"color": "cluster"},
                 title="The same segments, seen with the age")
fig.show()

**The clusters overlap much more in this plane: they were built on the income, which is not shown here. Still, a clear pattern appears: no customer older than 40 has a spending score above 60. The big spenders (clusters 1 and 4) are all young.**

10. Give a short name to each segment, from the profile table and the plots.

**For example (the cluster numbers depend on the run, read them in your own table):**

| cluster | income | spending | name |
|---|---|---|---|
| 0 | average (55 k\$) | average (50) | **standard customers**, the largest group (81) |
| 1 | high (87 k\$) | high (82) | **wealthy big spenders**, young (32.7) |
| 2 | high (88 k\$) | low (17) | **wealthy careful customers** (41.1) |
| 3 | low (26 k\$) | low (21) | **careful customers on a budget**, the oldest (45.2) |
| 4 | low (26 k\$) | high (79) | **young spenders on a budget**, the youngest (25.3) |

11. The marketing team has a budget for one campaign. Which segment(s) would you target, and how? Write two or three sentences.

**The wealthy careful customers (cluster 2) are the largest untapped potential: they have the money but spend little at the mall. A campaign that brings them in (premium brands, services, comfort) could raise their spending. The wealthy big spenders (cluster 1) are the most valuable today: a loyalty programme would help to keep them. The young spenders on a budget (cluster 4) spend a lot compared with their income: they are a good target for promotions, but pushing them to spend more raises ethical questions. Remember that a segment is only a summary: test the campaign on a sample before generalising.**

## To go further 🚀

12. Cluster the customers on the **three** numerical variables (age, income, spending score, standardised). Which $K$ does the silhouette score suggest? Compare the new clusters with the 5 segments with `pd.crosstab`. What changes?

In [ ]:
X3_scaled = StandardScaler().fit_transform(dataset[["Age", "Annual Income (k$)", "Spending Score (1-100)"]])
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X3_scaled)
    print(f"K={k:>2}: silhouette score {silhouette_score(X3_scaled, labels):.3f}")

dataset["cluster_3_variables"] = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(X3_scaled)
print(pd.crosstab(dataset["cluster"], dataset["cluster_3_variables"]))
dataset.groupby("cluster_3_variables")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)

K= 2: silhouette score 0.335
K= 3: silhouette score 0.358
K= 4: silhouette score 0.404
K= 5: silhouette score 0.417
K= 6: silhouette score 0.427
K= 7: silhouette score 0.417
K= 8: silhouette score 0.409
K= 9: silhouette score 0.421
K=10: silhouette score 0.409
cluster_3_variables   0   1   2   3   4   5
cluster                                    
0                     1  43   0  36   0   1
1                     0   0  39   0   0   0
2                    32   1   0   2   0   0
3                     0   1   0   0  21   1
4                     0   0   0   0   0  22


,Age,Annual Income (k$),Spending Score (1-100)
cluster_3_variables,,,
0,41.9,88.9,17.0
1,56.3,54.3,49.1
2,32.7,86.5,82.1
3,26.7,57.6,47.8
4,45.5,26.3,19.4
5,25.2,25.8,76.9


**With three variables, the silhouette scores are lower (at most 0.427, for $K=6$, against 0.555 with two variables) and less clear-cut ($K=5$ gives 0.417): the groups are less separated in 3 dimensions. With $K=6$, the corner segments mostly survive, but the large "standard" segment (cluster 0, 81 customers) is split by age, into an older group (43 customers, mean age 56.3) and a younger one (36 customers, mean age 26.7). Adding a variable changes the segments: the choice of the variables is part of the modelling, and it depends on the question you want to answer.**